# Segmentation

In [15]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

Chargement des données

In [16]:
df = pd.read_csv("../data/processed/transactions_nettoyees.csv")
df.head()

,transaction_id,customer_id,transaction_date,transaction_time,account_type,payment_method,direction,amount,balance_after,narration,...,day,hour,weekday,month_name,clean_narration,clean_merchant_name,is_debit,is_credit,transaction_type,time_period
0,TX0000001,CUST0001,2024-01-01,09:34:00,Courant,Virement instantané,Credit,11998.10,19846.02,virement employeur,...,1,9,Monday,January,VIREMENT EMPLOYEUR,SALAIRE,0,1,Revenu,Matin
1,TX0000003,CUST0001,2024-01-02,09:40:00,Courant,Wallet,Debit,21.55,19824.47,TRANSPORT CAREEM Rabat,...,2,9,Tuesday,January,TRANSPORT CAREEM RABAT,CAREEM,1,0,Dépense,Matin
2,TX0000002,CUST0001,2024-01-02,11:16:00,Courant,Carte,Debit,147.03,19677.44,CARBURANT SHELL Rabat,...,2,11,Tuesday,January,CARBURANT SHELL RABAT,SHELL,1,0,Dépense,Matin
3,TX0000004,CUST0001,2024-01-03,08:53:00,Courant,Carte,Debit,403.15,19274.29,JUMIA MAROC,...,3,8,Wednesday,January,JUMIA MAROC,JUMIA,1,0,Dépense,Matin
4,TX0000005,CUST0001,2024-01-03,18:49:00,Courant,Virement,Debit,775.01,18499.28,LOYER APPARTEMENT Rabat,...,3,18,Wednesday,January,LOYER APPARTEMENT RABAT,LOYER APPARTEMENT,1,0,Dépense,Soir


In [17]:
print(df.columns.tolist())

['transaction_id', 'customer_id', 'transaction_date', 'transaction_time', 'account_type', 'payment_method', 'direction', 'amount', 'balance_after', 'narration', 'merchant_name', 'merchant_city', 'merchant_country', 'transaction_datetime', 'year', 'month', 'day', 'hour', 'weekday', 'month_name', 'clean_narration', 'clean_merchant_name', 'is_debit', 'is_credit', 'transaction_type', 'time_period']


Fonction de nettoyage de texte

In [18]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9À-ÿ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

Nettoyer merchant et narration

In [19]:
df["merchant_name"] = df["merchant_name"].apply(clean_text)
df["narration"] = df["narration"].apply(clean_text)

df["text_to_segment"] = (df["merchant_name"].fillna("") + " " + df["narration"].fillna("")).str.strip()

df[["merchant_name", "narration", "text_to_segment"]].head(10)

,merchant_name,narration,text_to_segment
0,salaire,virement employeur,salaire virement employeur
1,careem,transport careem rabat,careem transport careem rabat
2,shell,carburant shell rabat,shell carburant shell rabat
3,jumia,jumia maroc,jumia jumia maroc
4,loyer appartement,loyer appartement rabat,loyer appartement loyer appartement rabat
5,atm,retrait gab tanger,atm retrait gab tanger
6,loyer appartement,vir loyer fès,loyer appartement vir loyer fès
7,carrefour,paiement carrefour rabat,carrefour paiement carrefour rabat
8,cinéma megarama,megarama rabat,cinéma megarama megarama rabat
9,orange,orange mobile,orange orange mobile


---  
### Définir les catégories et mots-clés

Dictionnaire de règles

In [20]:
CATEGORY_RULES = {
    "Retrait cash": [
        "atm", "guichet", "cash withdrawal", "retrait", "dab"
    ],
    "Revenu": [
        "salary", "salaire", "payroll", "virement recu", "transfer from employer",
        "versement", "depot", "remboursement", "freelance", "virement entrant", "famille"
    ],
    "Alimentation": [
        "carrefour", "marjane", "bim", "aswak", "attaqaddoum", "supermarket",
        "grocery", "epicerie", "market", "label vie", "restaurant","cafe",
        "coffee", "mcdonald", "kfc"
    ],
    "Transport": [
        "uber", "careem", "taxi", "tram", "train", "oncf", "station",
        "shell", "afriquia", "petrom", "total", "ola energy", "fuel", "gas", "indrive"
    ],
    "Factures": [
        "electricite", "eau", "water bill", "electric bill", "lydec", "redal",
        "orange", "inwi", "iam", "maroc telecom", "internet", "wifi", "phone bill"
    ],
    "Santé": [
        "pharmacie", "pharmacy", "clinique", "clinic", "hospital",
        "medecin", "doctor", "laboratoire", "lab"
    ],
    "Shopping": [
        "zara", "hm", "h&m", "bershka", "pull and bear", "pull&bear",
        "decathlon", "ikea", "store", "mall"
    ],
    "E-commerce": [
        "amazon", "jumia", "aliexpress", "paypal", "stripe", "shopify"
    ],
    "Loisirs": [
        "netflix", "spotify", "cinema", "movie", "gaming", "playstation",
        "xbox", "megarama"
    ],
    "Voyage": [
        "booking", "airbnb", "hotel", "ryanair", "royal air maroc",
        "flight", "travel", "agence voyage", "air arabia", "airarabia"
    ],
    "Logement": [
        "rent", "loyer", "syndic", "immobilier", "agency", "agence immobiliere"
    ]
}

---  
### Fonction de segmentation

Fonction principale

In [21]:
def assign_category(row):
    text = row["text_to_segment"]
    amount = row["amount"]

    # 1. Revenu : si montant positif + mots-clés
    revenue_keywords = CATEGORY_RULES["Revenu"]
    if amount > 0:
        if any(keyword in text for keyword in revenue_keywords):
            return "Revenu"

    # 2. Retrait cash
    for keyword in CATEGORY_RULES["Retrait cash"]:
        if keyword in text:
            return "Retrait cash"

    # 3. Revenu même sans montant positif explicite
    for keyword in CATEGORY_RULES["Revenu"]:
        if keyword in text:
            return "Revenu"

    # 4. Autres catégories de dépenses
    ordered_categories = [
        "Alimentation",
        "Transport",
        "Factures",
        "Santé",
        "Shopping",
        "E-commerce",
        "Loisirs",
        "Voyage",
        "Logement"
    ]

    for category in ordered_categories:
        for keyword in CATEGORY_RULES[category]:
            if keyword in text:
                return category

    return "Autres"

Appliquer la segmentation

In [22]:
df["category"] = df.apply(assign_category, axis=1)
df[["merchant_name", "narration", "amount", "category"]].head(20)

,merchant_name,narration,amount,category
0,salaire,virement employeur,11998.10,Revenu
1,careem,transport careem rabat,21.55,Transport
2,shell,carburant shell rabat,147.03,Transport
3,jumia,jumia maroc,403.15,E-commerce
4,loyer appartement,loyer appartement rabat,775.01,Logement
5,atm,retrait gab tanger,327.10,Retrait cash
6,loyer appartement,vir loyer fès,1336.85,Logement
7,carrefour,paiement carrefour rabat,199.14,Alimentation
8,cinéma megarama,megarama rabat,43.30,Loisirs
9,orange,orange mobile,133.73,Factures


---  
### Vérifier la qualité de la segmentation

Répartition globale

In [23]:
df["category"].value_counts(dropna=False)

category
Alimentation    8832
Transport       7100
Factures        4684
E-commerce      4193
Retrait cash    3763
Shopping        3738
Loisirs         3575
Logement        3265
Santé           3213
Revenu          2589
Voyage          1750
Name: count, dtype: int64

Pourcentage par catégorie

In [24]:
df["category"].value_counts(normalize=True).mul(100).round(2)

category
Alimentation    18.91
Transport       15.20
Factures        10.03
E-commerce       8.98
Retrait cash     8.06
Shopping         8.00
Loisirs          7.65
Logement         6.99
Santé            6.88
Revenu           5.54
Voyage           3.75
Name: proportion, dtype: float64

Vérifier les “Autres”

In [25]:
df_autres = df[df["category"] == "Autres"][
    ["merchant_name", "narration", "text_to_segment", "amount"]
].copy()

df_autres.head()

,merchant_name,narration,text_to_segment,amount


In [26]:
df_autres["text_to_segment"].value_counts().head(50)

Series([], Name: count, dtype: int64)

Vérifier catégorie par catégorie

In [27]:
df[df["category"] == "Alimentation"][
    ["merchant_name", "narration", "amount", "text_to_segment"]
].head(30)
# on reprend la même chose pour Transport, Factures, E-commerce, Revenu

,merchant_name,narration,amount,text_to_segment
7,carrefour,paiement carrefour rabat,199.14,carrefour paiement carrefour rabat
13,bim,bim store rabat,29.73,bim bim store rabat
25,marjane,achat marjane market rabat,352.75,marjane achat marjane market rabat
27,marjane,achat marjane market rabat,113.93,marjane achat marjane market rabat
29,marjane,cb marjane rabat,100.48,marjane cb marjane rabat
31,bim,paiement bim rabat,25.03,bim paiement bim rabat
33,carrefour,carrefour market rabat,220.21,carrefour carrefour market rabat
35,carrefour,carrefour market rabat,63.71,carrefour carrefour market rabat
36,bim,bim store rabat,15.60,bim bim store rabat
39,marjane,paiement marjane rabat,357.58,marjane paiement marjane rabat


---  
### Sauvegarder le résultat

Export

In [28]:
output_path = Path("../data/processed/transactions_segmentees.csv")
df.to_csv(output_path, index=False)
print(f"Fichier sauvegardé : {output_path}")

Fichier sauvegardé : ..\data\processed\transactions_segmentees.csv
